# Tarea: SQL y pandas con múltiples tablas

En esta tarea vas a trabajar con un caso de **academia / cursos online** usando varias tablas relacionadas.

## Objetivo
Resolver **10 ejercicios**, y para cada uno debes:

1. escribir la consulta en **SQL**
2. escribir la solución equivalente en **pandas**

## Reglas
- Intenta resolver primero en SQL.
- Luego haz la versión en pandas.
- Puedes crear celdas adicionales si las necesitas.
- La idea es que veas la equivalencia entre ambos enfoques.

## Tablas del caso
- `estudiantes`
- `cursos`
- `inscripciones`
- `pagos`

In [1]:
import pandas as pd
import sqlite3
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

## 1. Crear datos mock

In [2]:
estudiantes_data = [
    (1, "Ana Gómez", "Bogotá", "2024-01-15"),
    (2, "Luis Pérez", "Medellín", "2024-01-20"),
    (3, "Sofía Ramírez", "Cali", "2024-02-01"),
    (4, "Carlos Torres", "Bogotá", "2024-02-10"),
    (5, "Valentina Ruiz", "Barranquilla", "2024-02-15"),
    (6, "Mateo Castro", "Cartagena", "2024-03-01"),
    (7, "Laura Sánchez", "Bucaramanga", "2024-03-05"),
    (8, "Andrés Mejía", "Medellín", "2024-03-08"),
    (9, "Camila Herrera", "Cali", "2024-03-12"),
    (10, "Juan López", "Bogotá", "2024-03-18"),
    (11, "Paula Moreno", "Pereira", "2024-03-20"),
    (12, "Diego Vargas", "Manizales", "2024-03-22"),
]

cursos_data = [
    (101, "SQL desde cero", "Datos", 120, "Básico"),
    (102, "Pandas para análisis", "Datos", 150, "Intermedio"),
    (103, "Python básico", "Programación", 100, "Básico"),
    (104, "Visualización con Power BI", "BI", 180, "Intermedio"),
    (105, "Machine Learning Intro", "Datos", 220, "Avanzado"),
    (106, "Excel para negocios", "Oficina", 90, "Básico"),
    (107, "Estadística aplicada", "Datos", 160, "Intermedio"),
    (108, "Automatización con Python", "Programación", 200, "Avanzado"),
]

inscripciones_data = [
    (1001, 1, 101, "2024-04-01", "completado"),
    (1002, 1, 102, "2024-04-05", "activo"),
    (1003, 2, 101, "2024-04-02", "activo"),
    (1004, 2, 106, "2024-04-03", "completado"),
    (1005, 3, 103, "2024-04-04", "activo"),
    (1006, 3, 104, "2024-04-07", "cancelado"),
    (1007, 4, 101, "2024-04-08", "completado"),
    (1008, 4, 107, "2024-04-10", "activo"),
    (1009, 5, 102, "2024-04-09", "activo"),
    (1010, 5, 105, "2024-04-11", "activo"),
    (1011, 6, 106, "2024-04-12", "completado"),
    (1012, 6, 103, "2024-04-14", "activo"),
    (1013, 7, 101, "2024-04-15", "activo"),
    (1014, 7, 104, "2024-04-16", "activo"),
    (1015, 8, 102, "2024-04-17", "completado"),
    (1016, 8, 107, "2024-04-18", "activo"),
    (1017, 9, 103, "2024-04-19", "cancelado"),
    (1018, 9, 108, "2024-04-20", "activo"),
    (1019, 10, 101, "2024-04-21", "activo"),
    (1020, 10, 102, "2024-04-22", "activo"),
    (1021, 10, 107, "2024-04-22", "activo"),
    (1022, 11, 104, "2024-04-23", "completado"),
    (1023, 11, 106, "2024-04-24", "activo"),
    (1024, 12, 105, "2024-04-25", "activo"),
    (1025, 12, 108, "2024-04-26", "activo"),
]

pagos_data = [
    (5001, 1001, 120, "tarjeta", "2024-04-01"),
    (5002, 1002, 150, "pse", "2024-04-05"),
    (5003, 1003, 120, "tarjeta", "2024-04-02"),
    (5004, 1004, 90, "transferencia", "2024-04-03"),
    (5005, 1005, 100, "pse", "2024-04-04"),
    (5006, 1007, 120, "tarjeta", "2024-04-08"),
    (5007, 1008, 160, "pse", "2024-04-10"),
    (5008, 1009, 150, "tarjeta", "2024-04-09"),
    (5009, 1010, 220, "tarjeta", "2024-04-11"),
    (5010, 1011, 90, "transferencia", "2024-04-12"),
    (5011, 1012, 100, "pse", "2024-04-14"),
    (5012, 1013, 120, "tarjeta", "2024-04-15"),
    (5013, 1014, 180, "tarjeta", "2024-04-16"),
    (5014, 1015, 150, "pse", "2024-04-17"),
    (5015, 1016, 160, "tarjeta", "2024-04-18"),
    (5016, 1018, 200, "tarjeta", "2024-04-20"),
    (5017, 1019, 120, "pse", "2024-04-21"),
    (5018, 1020, 150, "tarjeta", "2024-04-22"),
    (5019, 1021, 160, "pse", "2024-04-22"),
    (5020, 1022, 180, "transferencia", "2024-04-23"),
    (5021, 1023, 90, "pse", "2024-04-24"),
    (5022, 1024, 220, "tarjeta", "2024-04-25"),
    (5023, 1025, 200, "tarjeta", "2024-04-26"),
]

## 2. Convertir a DataFrames

In [3]:
estudiantes = pd.DataFrame(estudiantes_data, columns=["estudiante_id", "nombre", "ciudad", "fecha_registro"])
cursos = pd.DataFrame(cursos_data, columns=["curso_id", "nombre_curso", "categoria", "precio", "nivel"])
inscripciones = pd.DataFrame(inscripciones_data, columns=["inscripcion_id", "estudiante_id", "curso_id", "fecha_inscripcion", "estado"])
pagos = pd.DataFrame(pagos_data, columns=["pago_id", "inscripcion_id", "monto", "metodo_pago", "fecha_pago"])

estudiantes["fecha_registro"] = pd.to_datetime(estudiantes["fecha_registro"])
inscripciones["fecha_inscripcion"] = pd.to_datetime(inscripciones["fecha_inscripcion"])
pagos["fecha_pago"] = pd.to_datetime(pagos["fecha_pago"])

display(estudiantes.head())
display(cursos.head())
display(inscripciones.head())
display(pagos.head())

,estudiante_id,nombre,ciudad,fecha_registro
0,1,Ana Gómez,Bogotá,2024-01-15
1,2,Luis Pérez,Medellín,2024-01-20
2,3,Sofía Ramírez,Cali,2024-02-01
3,4,Carlos Torres,Bogotá,2024-02-10
4,5,Valentina Ruiz,Barranquilla,2024-02-15


,curso_id,nombre_curso,categoria,precio,nivel
0,101,SQL desde cero,Datos,120,Básico
1,102,Pandas para análisis,Datos,150,Intermedio
2,103,Python básico,Programación,100,Básico
3,104,Visualización con Power BI,BI,180,Intermedio
4,105,Machine Learning Intro,Datos,220,Avanzado


,inscripcion_id,estudiante_id,curso_id,fecha_inscripcion,estado
0,1001,1,101,2024-04-01,completado
1,1002,1,102,2024-04-05,activo
2,1003,2,101,2024-04-02,activo
3,1004,2,106,2024-04-03,completado
4,1005,3,103,2024-04-04,activo


,pago_id,inscripcion_id,monto,metodo_pago,fecha_pago
0,5001,1001,120,tarjeta,2024-04-01
1,5002,1002,150,pse,2024-04-05
2,5003,1003,120,tarjeta,2024-04-02
3,5004,1004,90,transferencia,2024-04-03
4,5005,1005,100,pse,2024-04-04


## 3. Cargar tablas a SQLite

In [4]:
conn = sqlite3.connect(":memory:")

estudiantes.to_sql("estudiantes", conn, index=False, if_exists="replace")
cursos.to_sql("cursos", conn, index=False, if_exists="replace")
inscripciones.to_sql("inscripciones", conn, index=False, if_exists="replace")
pagos.to_sql("pagos", conn, index=False, if_exists="replace")

def run_sql(query):
    return pd.read_sql_query(query, conn)

## 4. Diagrama lógico

Relaciones:
- `estudiantes.estudiante_id = inscripciones.estudiante_id`
- `cursos.curso_id = inscripciones.curso_id`
- `inscripciones.inscripcion_id = pagos.inscripcion_id`

### Punto 1. Selección y filtro básico

Trae todos los estudiantes que viven en **Bogotá**.

**Entregable:**
- una consulta en SQL
- una solución equivalente en pandas

In [7]:
# Punto 1 - SQL
# Escribe aquí tu consulta
query_1 = """
SELECT *
FROM estudiantes
WHERE ciudad = 'Bogotá'
"""
sql_1 = run_sql(query_1)
sql_1

,estudiante_id,nombre,ciudad,fecha_registro
0,1,Ana Gómez,Bogotá,2024-01-15 00:00:00
1,4,Carlos Torres,Bogotá,2024-02-10 00:00:00
2,10,Juan López,Bogotá,2024-03-18 00:00:00


In [8]:
# Punto 1 - pandas
# Escribe aquí tu solución
estudiantes[estudiantes["ciudad"] == "Bogotá"].reset_index(drop=True)

,estudiante_id,nombre,ciudad,fecha_registro
0,1,Ana Gómez,Bogotá,2024-01-15
1,4,Carlos Torres,Bogotá,2024-02-10
2,10,Juan López,Bogotá,2024-03-18


### Punto 2. Ordenamiento

Trae los cursos ordenados de **mayor a menor precio** y muestra solo:
- `nombre_curso`
- `categoria`
- `precio`

**Entregable:**
- SQL
- pandas

In [9]:
# Punto 2 - SQL
# Escribe aquí tu consulta
query_2 = """
SELECT nombre_curso, categoria, precio
FROM cursos
ORDER BY precio DESC
"""
sql_2 = run_sql(query_2)
sql_2

,nombre_curso,categoria,precio
0,Machine Learning Intro,Datos,220
1,Automatización con Python,Programación,200
2,Visualización con Power BI,BI,180
3,Estadística aplicada,Datos,160
4,Pandas para análisis,Datos,150
5,SQL desde cero,Datos,120
6,Python básico,Programación,100
7,Excel para negocios,Oficina,90


In [10]:
# Punto 2 - pandas
# Escribe aquí tu solución
cursos[["nombre_curso", "categoria","precio"]].sort_values("precio", ascending=False).reset_index(drop = True)

,nombre_curso,categoria,precio
0,Machine Learning Intro,Datos,220
1,Automatización con Python,Programación,200
2,Visualización con Power BI,BI,180
3,Estadística aplicada,Datos,160
4,Pandas para análisis,Datos,150
5,SQL desde cero,Datos,120
6,Python básico,Programación,100
7,Excel para negocios,Oficina,90


### Punto 3. Filtro con múltiples condiciones

Trae las inscripciones cuyo estado sea **activo** y cuya fecha de inscripción sea posterior al **2024-04-15**.

**Entregable:**
- SQL
- pandas

In [11]:
# Punto 3 - SQL
# Escribe aquí tu consulta
query_3 = """
SELECT *
FROM inscripciones
WHERE estado = 'activo' AND fecha_inscripcion > '2024-04-15'
"""
sql_3 = run_sql(query_3)
sql_3

,inscripcion_id,estudiante_id,curso_id,fecha_inscripcion,estado
0,1013,7,101,2024-04-15 00:00:00,activo
1,1014,7,104,2024-04-16 00:00:00,activo
2,1016,8,107,2024-04-18 00:00:00,activo
3,1018,9,108,2024-04-20 00:00:00,activo
4,1019,10,101,2024-04-21 00:00:00,activo
5,1020,10,102,2024-04-22 00:00:00,activo
6,1021,10,107,2024-04-22 00:00:00,activo
7,1023,11,106,2024-04-24 00:00:00,activo
8,1024,12,105,2024-04-25 00:00:00,activo
9,1025,12,108,2024-04-26 00:00:00,activo


In [12]:
# Punto 3 - pandas
# Escribe aquí tu solución
cond_estado = inscripciones["estado"] == "activo"
cond_fecha = inscripciones["fecha_inscripcion"] > "2024-04-15"
inscripciones_filtradas_pandas = inscripciones[cond_estado & cond_fecha].reset_index(drop=True)
inscripciones_filtradas_pandas

,inscripcion_id,estudiante_id,curso_id,fecha_inscripcion,estado
0,1014,7,104,2024-04-16,activo
1,1016,8,107,2024-04-18,activo
2,1018,9,108,2024-04-20,activo
3,1019,10,101,2024-04-21,activo
4,1020,10,102,2024-04-22,activo
5,1021,10,107,2024-04-22,activo
6,1023,11,106,2024-04-24,activo
7,1024,12,105,2024-04-25,activo
8,1025,12,108,2024-04-26,activo


### Punto 4. Agregación simple

Calcula:
- número total de cursos
- precio promedio
- precio mínimo
- precio máximo

Usa la tabla `cursos`.

**Entregable:**
- SQL
- pandas

In [13]:
# Punto 4 - SQL
# Escribe aquí tu consulta
query_4 = """
SELECT COUNT(curso_id) AS total_cursos,
       AVG(precio) AS precio_promedio,
       MIN(precio) AS precio_minimo,
       MAX(precio) AS precio_maximo
FROM cursos
"""
sql_4 = run_sql(query_4)
sql_4

,total_cursos,precio_promedio,precio_minimo,precio_maximo
0,8,152.5,90,220


In [14]:
# Punto 4 - pandas
# Escribe aquí tu solución
pandas_4 = pd.DataFrame({
    'total_cursos': [cursos['curso_id'].count()],
    'precio_promedio': [cursos['precio'].mean()],
    'precio_minimo': [cursos['precio'].min()],
    'precio_maximo': [cursos['precio'].max()]
})
pandas_4

,total_cursos,precio_promedio,precio_minimo,precio_maximo
0,8,152.5,90,220


### Punto 5. GROUP BY

Cuenta cuántos cursos hay por `categoria`.

Ordena de mayor a menor cantidad.

**Entregable:**
- SQL
- pandas

In [15]:
# Punto 5 - SQL
# Escribe aquí tu consulta
query_5 = """
SELECT categoria, COUNT(curso_id) AS cantidad_cursos
FROM cursos
GROUP BY categoria
ORDER BY cantidad_cursos DESC
"""
sql_5 = run_sql(query_5)
sql_5

,categoria,cantidad_cursos
0,Datos,4
1,Programación,2
2,Oficina,1
3,BI,1


In [16]:
# Punto 5 - pandas
# Escribe aquí tu solución
cursos.groupby("categoria")["curso_id"].count().sort_values(ascending=False).reset_index(name="cantidad_cursos")

,categoria,cantidad_cursos
0,Datos,4
1,Programación,2
2,BI,1
3,Oficina,1


### Punto 6. JOIN entre tablas

Muestra:
- nombre del estudiante
- nombre del curso
- estado de la inscripción

Usa `estudiantes`, `inscripciones` y `cursos`.

**Entregable:**
- SQL
- pandas

In [23]:
# Punto 6 - SQL
# Escribe aquí tu consulta
query_6 = """
SELECT e.nombre AS nombre_estudiante,
       c.nombre_curso,
       i.estado
FROM estudiantes e
JOIN inscripciones i ON e.estudiante_id = i.estudiante_id
JOIN cursos c ON i.curso_id = c.curso_id
"""
sql_6 = run_sql(query_6)
display(sql_6)

,nombre_estudiante,nombre_curso,estado
0,Ana Gómez,SQL desde cero,completado
1,Ana Gómez,Pandas para análisis,activo
2,Luis Pérez,SQL desde cero,activo
3,Luis Pérez,Excel para negocios,completado
4,Sofía Ramírez,Python básico,activo
5,Sofía Ramírez,Visualización con Power BI,cancelado
6,Carlos Torres,SQL desde cero,completado
7,Carlos Torres,Estadística aplicada,activo
8,Valentina Ruiz,Pandas para análisis,activo
9,Valentina Ruiz,Machine Learning Intro,activo


In [21]:
# Punto 6 - pandas
# Escribe aquí tu solución
pandas_6 = estudiantes.merge(inscripciones, on='estudiante_id', how='inner') \
                       .merge(cursos, on='curso_id', how='inner')
pandas_6 = pandas_6[['nombre', 'nombre_curso', 'estado']].rename(columns={'nombre': 'nombre_estudiante'})
display(pandas_6)

,nombre_estudiante,nombre_curso,estado
0,Ana Gómez,SQL desde cero,completado
1,Ana Gómez,Pandas para análisis,activo
2,Luis Pérez,SQL desde cero,activo
3,Luis Pérez,Excel para negocios,completado
4,Sofía Ramírez,Python básico,activo
5,Sofía Ramírez,Visualización con Power BI,cancelado
6,Carlos Torres,SQL desde cero,completado
7,Carlos Torres,Estadística aplicada,activo
8,Valentina Ruiz,Pandas para análisis,activo
9,Valentina Ruiz,Machine Learning Intro,activo


### Punto 7. LEFT JOIN

Trae todos los estudiantes, incluso si no tuvieran inscripciones, mostrando:
- `estudiante_id`
- `nombre`
- `inscripcion_id`

**Entregable:**
- SQL
- pandas

> Pista: en este dataset puede que todos tengan al menos una inscripción, pero igual debes practicar el `LEFT JOIN`.

In [25]:
# Punto 7 - SQL
# Escribe aquí tu consulta
query_7 = """
SELECT e.estudiante_id,
       e.nombre,
       i.inscripcion_id
FROM estudiantes e
LEFT JOIN inscripciones i ON e.estudiante_id = i.estudiante_id
"""
sql_7 = run_sql(query_7)
display(sql_7)

,estudiante_id,nombre,inscripcion_id
0,1,Ana Gómez,1001
1,1,Ana Gómez,1002
2,2,Luis Pérez,1003
3,2,Luis Pérez,1004
4,3,Sofía Ramírez,1005
5,3,Sofía Ramírez,1006
6,4,Carlos Torres,1007
7,4,Carlos Torres,1008
8,5,Valentina Ruiz,1009
9,5,Valentina Ruiz,1010


In [27]:
# Punto 7 - pandas
# Escribe aquí tu solución
pandas_7 = estudiantes.merge(inscripciones, on='estudiante_id', how='left')
pandas_7 = pandas_7[['estudiante_id', 'nombre', 'inscripcion_id']]
display(pandas_7)


,estudiante_id,nombre,inscripcion_id
0,1,Ana Gómez,1001
1,1,Ana Gómez,1002
2,2,Luis Pérez,1003
3,2,Luis Pérez,1004
4,3,Sofía Ramírez,1005
5,3,Sofía Ramírez,1006
6,4,Carlos Torres,1007
7,4,Carlos Torres,1008
8,5,Valentina Ruiz,1009
9,5,Valentina Ruiz,1010


### Punto 8. HAVING

Calcula cuántas inscripciones tiene cada estudiante y muestra solo aquellos con **2 o más**.

Debes mostrar:
- `estudiante_id`
- `nombre`
- `n_inscripciones`

**Entregable:**
- SQL
- pandas

In [29]:
# Punto 8 - SQL
# Escribe aquí tu consulta
query_8 = """
SELECT e.estudiante_id,
       e.nombre,
       COUNT(i.inscripcion_id) AS n_inscripciones
FROM estudiantes e
JOIN inscripciones i ON e.estudiante_id = i.estudiante_id
GROUP BY e.estudiante_id, e.nombre
HAVING COUNT(i.inscripcion_id) >= 2
"""
sql_8 = run_sql(query_8)
display(sql_8)

,estudiante_id,nombre,n_inscripciones
0,1,Ana Gómez,2
1,2,Luis Pérez,2
2,3,Sofía Ramírez,2
3,4,Carlos Torres,2
4,5,Valentina Ruiz,2
5,6,Mateo Castro,2
6,7,Laura Sánchez,2
7,8,Andrés Mejía,2
8,9,Camila Herrera,2
9,10,Juan López,3


In [30]:
# Punto 8 - pandas
# Escribe aquí tu solución
pandas_8 = estudiantes.merge(inscripciones, on='estudiante_id', how='inner')
pandas_8 = pandas_8.groupby(['estudiante_id', 'nombre'])['inscripcion_id'].count().reset_index(name='n_inscripciones')
pandas_8 = pandas_8[pandas_8['n_inscripciones'] >= 2]
display(pandas_8)

,estudiante_id,nombre,n_inscripciones
0,1,Ana Gómez,2
1,2,Luis Pérez,2
2,3,Sofía Ramírez,2
3,4,Carlos Torres,2
4,5,Valentina Ruiz,2
5,6,Mateo Castro,2
6,7,Laura Sánchez,2
7,8,Andrés Mejía,2
8,9,Camila Herrera,2
9,10,Juan López,3


### Punto 9. CASE WHEN / segmentación

Clasifica los cursos según su precio:
- `Bajo` si el precio es menor a 120
- `Medio` si el precio está entre 120 y 180 inclusive
- `Alto` si el precio es mayor a 180

Debes mostrar:
- `nombre_curso`
- `precio`
- `segmento_precio`

**Entregable:**
- SQL
- pandas

In [31]:
# Punto 9 - SQL
# Escribe aquí tu consulta
query_9 = """
SELECT nombre_curso,
       precio,
       CASE
           WHEN precio < 120 THEN 'Bajo'
           WHEN precio BETWEEN 120 AND 180 THEN 'Medio'
           ELSE 'Alto'
       END AS segmento_precio
FROM cursos
"""
sql_9 = run_sql(query_9)
display(sql_9)

,nombre_curso,precio,segmento_precio
0,SQL desde cero,120,Medio
1,Pandas para análisis,150,Medio
2,Python básico,100,Bajo
3,Visualización con Power BI,180,Medio
4,Machine Learning Intro,220,Alto
5,Excel para negocios,90,Bajo
6,Estadística aplicada,160,Medio
7,Automatización con Python,200,Alto


In [34]:
# Punto 9 - pandas
# Escribe aquí tu solución
def segmentar_precio(precio):
    if precio < 120:
        return 'Bajo'
    elif 120 <= precio <= 180:
        return 'Medio'
    else:
        return 'Alto'

pandas_9 = cursos[['nombre_curso', 'precio']].copy()
pandas_9['segmento_precio'] = pandas_9['precio'].apply(segmentar_precio)
display(pandas_9)

,nombre_curso,precio,segmento_precio
0,SQL desde cero,120,Medio
1,Pandas para análisis,150,Medio
2,Python básico,100,Bajo
3,Visualización con Power BI,180,Medio
4,Machine Learning Intro,220,Alto
5,Excel para negocios,90,Bajo
6,Estadística aplicada,160,Medio
7,Automatización con Python,200,Alto


### Punto 10. Ejercicio final integrador

Calcula el total pagado por categoría de curso.

Debes:
- unir `cursos`, `inscripciones` y `pagos`
- agrupar por `categoria`
- sumar `monto`
- ordenar de mayor a menor

**Entregable:**
- SQL
- pandas

In [36]:
# Punto 10 - SQL
# Escribe aquí tu consulta
query_10 = """
SELECT c.categoria,
       SUM(p.monto) AS total_pagado
FROM cursos c
JOIN inscripciones i ON c.curso_id = i.curso_id
JOIN pagos p ON i.inscripcion_id = p.inscripcion_id
GROUP BY c.categoria
ORDER BY total_pagado DESC
"""
sql_10 = run_sql(query_10)
display(sql_10)

,categoria,total_pagado
0,Datos,2120
1,Programación,600
2,BI,360
3,Oficina,270


In [38]:
# Punto 10 - pandas
# Escribe aquí tu solución
pandas_10 = cursos.merge(inscripciones, on='curso_id', how='inner') \
                  .merge(pagos, on='inscripcion_id', how='inner')
pandas_10 = pandas_10.groupby('categoria')['monto'].sum().reset_index(name='total_pagado') \
                  .sort_values(by='total_pagado', ascending=False)
display(pandas_10)

,categoria,total_pagado
1,Datos,2120
3,Programación,600
0,BI,360
2,Oficina,270


## 5. Espacio opcional para validaciones

Puedes usar esta sección para probar queries, revisar tablas o comparar resultados.

In [ ]:
# Ejemplos:
# display(estudiantes)
# display(cursos)
# display(inscripciones)
# display(pagos)

# query = """
# SELECT *
# FROM estudiantes
# """
# display(run_sql(query))